# Minh họa kiến trúc Thế hệ 2: Mạng nơ-ron hồi quy (Seq2Seq LSTM + Attention)

Mục tiêu của file này là minh họa cấu trúc cốt lõi của một hệ thống Tóm tắt văn bản Thế hệ 2 trên **dữ liệu thực tế** (500 bài báo y khoa tiếng Việt).

**Tại sao Thế hệ 2 bị thay thế?**
1. LSTM cần hàng triệu cặp câu và hàng tuần train trên siêu máy tính để đạt chất lượng tốt.
2. Tính toán tuần tự (từng từ một) → cực kỳ chậm với đoạn văn dài.
3. Hay bị 'quên' (Vanishing Gradient) khi văn bản quá dài.

=> Đây là lý do người ta chuyển sang **Thế hệ 3 & 4 (Transformer)** như PhoBERT, ViT5.

In [2]:
%pip install -qq torch pandas numpy
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pandas as pd
import numpy as np
import random
import re
import os
import glob
from pathlib import Path

DATA_DIR = '../data/vietnamese-summarization'
# Kiểm tra sự tồn tại của thư mục
if os.path.exists(DATA_DIR):
    print(f"Thư mục tồn tại: {DATA_DIR}")
    files = os.listdir(DATA_DIR)
    print(f"Danh sách file trong thư mục ({len(files)} file):")
    for f in files[:20]:  # Liệt kê tối đa 20 file đầu tiên
        print(f" - {f}")
    if len(files) > 20:
        print(" ... và các file khác.")
else:
    print(f"LỖI: Thư mục không tồn tại: {DATA_DIR}")
    # Kiểm tra các thư mục cha để xem sai từ đoạn nào
    parent = "/content/drive/MyDrive"
    if os.path.exists(parent):
        print(f"Thư mục con của {parent}: {os.listdir(parent)}")

# Tự động phát hiện môi trường: Nếu có GPU (như trên Google Colab) thì dùng cuda, ngược lại dùng cpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Đang sử dụng thiết bị:", device)

Note: you may need to restart the kernel to use updated packages.
Thư mục tồn tại: ../../data/vietnamese-summarization
Danh sách file trong thư mục (9 file):
 - herding_bio_medicine.csv
 - Kmeans_1024_new.csv
 - bio_medicine.csv
 - herding_512_bio_medicine.csv
 - herding_prompt_512_bio_medicine.csv
 - Kmeans_512_token_new.csv
 - Kmeans_512_new.csv
 - train
 - test
Đang sử dụng thiết bị: cuda


## 1. Khối Mã hóa (Encoder)
Encoder đọc **tuần tự** từng từ trong câu gốc, cập nhật Trạng thái ẩn (Hidden State h, c) và tích lũy ngữ cảnh toàn câu để truyền cho Decoder.

In [3]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size
        # Embedding: ánh xạ ID từ → vector số thực
        self.embedding = nn.Embedding(input_size, hidden_size)
        # LSTM: ghi nhớ thông tin dài hạn tốt hơn RNN thuần
        self.lstm = nn.LSTM(hidden_size, hidden_size)

    def forward(self, input, hidden):
        embedded = self.embedding(input).view(1, 1, -1)
        output, hidden = self.lstm(embedded, hidden)
        return output, hidden

    def initHidden(self):
        # Khởi tạo cặp trạng thái (h_0, c_0) bằng vector 0
        return (torch.zeros(1, 1, self.hidden_size, device=device),
                torch.zeros(1, 1, self.hidden_size, device=device))

## 2. Khối Giải mã có Cơ chế Chú ý (AttnDecoderRNN)
Decoder nhả ra từng từ của bản tóm tắt. Nhờ **Attention**, khi nhả một từ, nó biết phải 'nhìn lại' vào phần nào quan trọng nhất của câu gốc.

**Attention Masking:** Để attention không nhìn vào các vector 0 (phần padding), hàm `forward` nhận thêm `attn_mask` và đặt trọng số attention tại vị trí padding về `-inf` trước khi qua softmax.

In [4]:
class AttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, max_length=10):
        super(AttnDecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.max_length = max_length

        self.embedding = nn.Embedding(self.output_size, self.hidden_size)
        # Lớp tuyến tính tính điểm Attention (score) cho mỗi vị trí encoder
        self.attn = nn.Linear(self.hidden_size * 2, self.max_length)
        self.attn_combine = nn.Linear(self.hidden_size * 2, self.hidden_size)
        self.lstm = nn.LSTM(self.hidden_size, self.hidden_size)
        self.out = nn.Linear(self.hidden_size, self.output_size)

    def forward(self, input, hidden, encoder_outputs, attn_mask=None):
        embedded = self.embedding(input).view(1, 1, -1)

        # Tính điểm attention thô
        attn_scores = self.attn(torch.cat((embedded[0], hidden[0][0]), 1))  # [1, max_length]

        # --- FIX: Mask padding – đặt vị trí padding về -inf trước softmax ---
        if attn_mask is not None:
            # attn_mask: 1 = vị trí có từ thực, 0 = padding
            attn_scores = attn_scores.masked_fill(attn_mask.unsqueeze(0) == 0, float('-inf'))

        attn_weights = F.softmax(attn_scores, dim=1)  # [1, max_length]

        # Áp dụng attention lên encoder_outputs
        attn_applied = torch.bmm(attn_weights.unsqueeze(0),
                                  encoder_outputs.unsqueeze(0))  # [1, 1, hidden]

        output = torch.cat((embedded[0], attn_applied[0]), 1)
        output = self.attn_combine(output).unsqueeze(0)
        output = F.relu(output)
        output, hidden = self.lstm(output, hidden)
        output = F.log_softmax(self.out(output[0]), dim=1)
        return output, hidden, attn_weights

    def initHidden(self):
        return (torch.zeros(1, 1, self.hidden_size, device=device),
                torch.zeros(1, 1, self.hidden_size, device=device))

## 3. Tiền xử lý Dữ liệu (Data Preprocessing)

In [5]:
SOS_token = 0
EOS_token = 1
UNK_token = 2

class Lang:
    """Từ điển hai chiều: từ ↔ số ID."""
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: 'SOS', 1: 'EOS', 2: 'UNK'}
        self.n_words = 3

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

def normalizeString(s):
    """Chuẩn hóa văn bản tiếng Việt.
    GIỮ LẠI chữ số vì số liệu y khoa (tuổi, liều lượng, ngày tháng) quan trọng."""
    s = str(s).lower().strip()
    s = re.sub(r'([.!?])', r' \1', s)
    # Giữ lại chữ cái a-z, tiếng Việt, và chữ số 0-9
    s = re.sub(r'[^a-zA-Z0-9àáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ ]+', ' ', s)
    return s.strip()

def load_all_csv_files(data_dir):
    """Nạp toàn bộ CSV trong thư mục và chuẩn hóa schema về content/summary/category."""
    csv_files = sorted(data_dir.glob('*.csv'))
    if not csv_files:
        raise FileNotFoundError(f'Không có file CSV nào trong {data_dir}')

    frames = []
    for csv_file in csv_files:
        frame = pd.read_csv(csv_file)
        frame = frame.rename(columns={
            'Document': 'content',
            'Summary': 'summary',
            'Dataset': 'category',
        })
        required_cols = ['content', 'summary', 'category']
        missing_cols = [col for col in required_cols if col not in frame.columns]
        if missing_cols:
            raise ValueError(f'{csv_file.name} thiếu cột: {missing_cols}')

        frame = frame[required_cols].copy()
        frame['source_file'] = csv_file.name
        frames.append(frame)
        print(f'  - {csv_file.name}: {len(frame):,} dòng')

    df_all = pd.concat(frames, ignore_index=True)
    before_clean = len(df_all)
    df_all = df_all.dropna(subset=['content', 'summary'])
    df_all = df_all[df_all['content'].astype(str).str.strip().ne('')]
    df_all = df_all[df_all['summary'].astype(str).str.strip().ne('')]
    df_all = df_all.drop_duplicates(subset=['content', 'summary']).reset_index(drop=True)
    print(f'Tổng sau khi gộp/làm sạch/bỏ trùng: {len(df_all):,}/{before_clean:,} dòng')
    return df_all

print('Bắt đầu nạp toàn bộ file CSV...')
DATA_DIR = Path('../data/train')
print('Thư mục data:', DATA_DIR)
df = load_all_csv_files(DATA_DIR)

# None = train toàn bộ dữ liệu sau khi gộp. Đặt số nguyên, ví dụ 500, nếu muốn chạy demo nhanh.
MAX_SAMPLES = None
df_train = df if MAX_SAMPLES is None else df.iloc[:MAX_SAMPLES].copy()
TRAIN_SIZE = len(df_train)

if TRAIN_SIZE < len(df):
    test_row = df.iloc[TRAIN_SIZE]
    test_note = 'mẫu kế tiếp, chưa nằm trong tập train'
else:
    test_row = df_train.iloc[-1]
    test_note = 'mẫu cuối trong tập train vì đang dùng toàn bộ dữ liệu để train'

# Thống kê tỷ lệ bài bị cắt
token_lengths = df_train['content'].apply(lambda x: len(normalizeString(x).split()))
print(f'Số mẫu train: {TRAIN_SIZE:,}')
print(f'Độ dài content – median: {token_lengths.median():.0f}, max: {token_lengths.max()}')
print(f'Số bài dài > 500 token: {(token_lengths > 500).sum():,}/{TRAIN_SIZE:,} ({(token_lengths > 500).mean()*100:.1f}%)')
print('Mẫu dùng để kiểm thử:', test_note)

# Xây dựng từ điển và danh sách cặp câu
lang  = Lang('vietnamese')
pairs = []
for _, row in df_train.iterrows():
    content = normalizeString(row['content'])
    summary = normalizeString(row['summary'])
    lang.addSentence(content)
    lang.addSentence(summary)
    pairs.append((content, summary))

print(f'Từ điển: {lang.n_words} từ duy nhất.')

Bắt đầu nạp toàn bộ file CSV...
Thư mục data: ../../data/vietnamese-summarization/train
  - Kmeans_1024_new.csv: 1,000 dòng
  - Kmeans_512_new.csv: 1,000 dòng
  - Kmeans_512_token_new.csv: 1,000 dòng
  - bio_medicine.csv: 1,000 dòng
  - herding_512_bio_medicine.csv: 1,000 dòng
  - herding_bio_medicine.csv: 1,000 dòng
  - herding_prompt_512_bio_medicine.csv: 1,000 dòng
Tổng sau khi gộp/làm sạch/bỏ trùng: 3,182/7,000 dòng
Số mẫu train: 3,182
Độ dài content – median: 439, max: 1420
Số bài dài > 500 token: 1,298/3,182 (40.8%)
Mẫu dùng để kiểm thử: mẫu cuối trong tập train vì đang dùng toàn bộ dữ liệu để train
Từ điển: 7590 từ duy nhất.


## 4. Huấn luyện (Training)
Hàm `train()` thực hiện một bước forward + backward:
- **Teacher Forcing** (50%): feed đúng nhãn target làm input bước tiếp → học nhanh hơn.
- **Tính loss qua toàn bộ target**: không `break` sớm khi dự đoán EOS → phạt đầy đủ.
- **Attention Mask**: chỉ cho attention nhìn vào vị trí có từ thực của encoder.

In [6]:
MAX_LENGTH          = 500   # Tăng lên 500 để giảm tỷ lệ bài bị cắt
TEACHER_FORCING_RATIO = 0.5  # 50% bước dùng Teacher Forcing

# ── Hàm chuyển đổi văn bản → Tensor ──────────────────────────────
def indexesFromSentence(lang, sentence):
    return [lang.word2index.get(w, UNK_token) for w in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence) + [EOS_token]
    if len(indexes) > MAX_LENGTH:
        indexes = indexes[:MAX_LENGTH - 1] + [EOS_token]
    return torch.tensor(indexes, dtype=torch.long, device=device).view(-1, 1)

def tensorsFromPair(pair):
    return tensorFromSentence(lang, pair[0]), tensorFromSentence(lang, pair[1])

# ── Hàm train một cặp câu ─────────────────────────────────────────
def train(input_tensor, target_tensor, encoder, decoder,
          encoder_optimizer, decoder_optimizer, criterion):
    encoder_hidden = encoder.initHidden()
    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()

    input_length  = input_tensor.size(0)
    target_length = target_tensor.size(0)

    encoder_outputs = torch.zeros(MAX_LENGTH, encoder.hidden_size, device=device)
    loss = 0

    # Encoder đọc từng từ một
    for ei in range(input_length):
        enc_out, encoder_hidden = encoder(input_tensor[ei], encoder_hidden)
        encoder_outputs[ei] = enc_out[0, 0]

    # Tạo attention mask: 1 = vị trí có từ thực, 0 = padding
    attn_mask = torch.zeros(MAX_LENGTH, device=device)
    attn_mask[:input_length] = 1

    decoder_input  = torch.tensor([[SOS_token]], device=device)
    decoder_hidden = encoder_hidden
    use_teacher_forcing = random.random() < TEACHER_FORCING_RATIO

    # Decoder nhả từng từ, tính loss qua TOÀN BỘ target (không break sớm)
    for di in range(target_length):
        dec_out, decoder_hidden, _ = decoder(
            decoder_input, decoder_hidden, encoder_outputs, attn_mask)
        loss += criterion(dec_out, target_tensor[di])

        if use_teacher_forcing:
            decoder_input = target_tensor[di]          # Feed nhãn đúng
        else:
            _, topi = dec_out.topk(1)
            decoder_input = topi.squeeze().detach()    # Feed dự đoán (không break)

    loss.backward()
    encoder_optimizer.step()
    decoder_optimizer.step()
    return loss.item() / target_length

# ── Khởi tạo mô hình & Optimizer ─────────────────────────────────
hidden_size = 256
encoder = EncoderRNN(lang.n_words, hidden_size).to(device)
decoder = AttnDecoderRNN(hidden_size, lang.n_words, max_length=MAX_LENGTH).to(device)

encoder_optimizer = optim.SGD(encoder.parameters(), lr=0.01)
decoder_optimizer = optim.SGD(decoder.parameters(), lr=0.01)
criterion = nn.NLLLoss()

# ── Vòng lặp huấn luyện ──────────────────────────────────────────
print(f'Bắt đầu huấn luyện với {TRAIN_SIZE:,} bài báo, {5} epochs...')
for epoch in range(5):
    total_loss = 0
    random.shuffle(pairs)
    for i, pair in enumerate(pairs):
        inp_t, tgt_t = tensorsFromPair(pair)
        loss = train(inp_t, tgt_t, encoder, decoder,
                     encoder_optimizer, decoder_optimizer, criterion)
        total_loss += loss
        if (i + 1) % 100 == 0:
            avg = total_loss / 100
            print(f'Epoch {epoch+1} | Bài {i+1:,}/{TRAIN_SIZE:,} | Loss TB: {avg:.4f}')
            total_loss = 0
print('Hoàn tất huấn luyện!')

Bắt đầu huấn luyện với 3,182 bài báo, 5 epochs...
Epoch 1 | Bài 100/3,182 | Loss TB: 7.8034
Epoch 1 | Bài 200/3,182 | Loss TB: 7.1911
Epoch 1 | Bài 300/3,182 | Loss TB: 6.7882
Epoch 1 | Bài 400/3,182 | Loss TB: 6.7568
Epoch 1 | Bài 500/3,182 | Loss TB: 6.5687
Epoch 1 | Bài 600/3,182 | Loss TB: 6.5289
Epoch 1 | Bài 700/3,182 | Loss TB: 6.4108
Epoch 1 | Bài 800/3,182 | Loss TB: 6.3992
Epoch 1 | Bài 900/3,182 | Loss TB: 6.4416
Epoch 1 | Bài 1,000/3,182 | Loss TB: 6.3591
Epoch 1 | Bài 1,100/3,182 | Loss TB: 6.2713
Epoch 1 | Bài 1,200/3,182 | Loss TB: 6.4275
Epoch 1 | Bài 1,300/3,182 | Loss TB: 6.3120
Epoch 1 | Bài 1,400/3,182 | Loss TB: 6.3014
Epoch 1 | Bài 1,500/3,182 | Loss TB: 6.2004
Epoch 1 | Bài 1,600/3,182 | Loss TB: 6.3067
Epoch 1 | Bài 1,700/3,182 | Loss TB: 6.2218
Epoch 1 | Bài 1,800/3,182 | Loss TB: 6.2756
Epoch 1 | Bài 1,900/3,182 | Loss TB: 6.3005
Epoch 1 | Bài 2,000/3,182 | Loss TB: 6.1975
Epoch 1 | Bài 2,100/3,182 | Loss TB: 6.2225
Epoch 1 | Bài 2,200/3,182 | Loss TB: 6.2609


## 5. Kiểm thử trên Data thật (Testing / Inference)

In [10]:
def evaluate(encoder, decoder, sentence):
    """Chạy suy luận: câu văn → câu tóm tắt (không dùng Teacher Forcing)."""
    with torch.no_grad():
        input_tensor = tensorFromSentence(lang, sentence)
        input_length = input_tensor.size(0)
        encoder_hidden = encoder.initHidden()
        encoder_outputs = torch.zeros(MAX_LENGTH, encoder.hidden_size, device=device)

        for ei in range(input_length):
            enc_out, encoder_hidden = encoder(input_tensor[ei], encoder_hidden)
            encoder_outputs[ei] = enc_out[0, 0]

        # Tạo attention mask cho inference
        attn_mask = torch.zeros(MAX_LENGTH, device=device)
        attn_mask[:input_length] = 1

        decoder_input  = torch.tensor([[SOS_token]], device=device)
        decoder_hidden = encoder_hidden
        decoded_words  = []

        for _ in range(MAX_LENGTH):
            dec_out, decoder_hidden, _ = decoder(
                decoder_input, decoder_hidden, encoder_outputs, attn_mask)
            _, topi = dec_out.data.topk(1)
            if topi.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(lang.index2word.get(topi.item(), 'UNK'))
            decoder_input = topi.squeeze().detach()

        return ' '.join(decoded_words)

TEST_DIR = Path('../data/test')
df_test = load_all_csv_files(TEST_DIR)

print(f"Đang chạy kiểm thử trên TOÀN BỘ {len(df_test)} bài báo của tập Test...")
results = []

for idx, row in df_test.iterrows():
    content = normalizeString(row['content'])
    summary = normalizeString(row['summary'])
    pred = evaluate(encoder, decoder, content)
        
    results.append({'content': content, 'summary': summary, 'prediction': pred})

df_results = pd.DataFrame(results)
df_results.to_csv('test_predictions.csv', index=False)
print(f"\nĐã hoàn tất! Toàn bộ {len(df_test)} kết quả đã được lưu vào file 'test_predictions.csv'.")

  - Kmeans_1024_new.csv: 100 dòng
  - Kmeans_512_new.csv: 100 dòng
  - Kmeans_512_token_new.csv: 100 dòng
  - bio_medicine.csv: 100 dòng
  - herding_512_bio_medicine.csv: 100 dòng
  - herding_bio_medicine.csv: 100 dòng
  - herding_prompt_512_bio_medicine.csv: 100 dòng
Tổng sau khi gộp/làm sạch/bỏ trùng: 292/700 dòng
Đang chạy kiểm thử trên TOÀN BỘ 292 bài báo của tập Test...

--- BÀI TEST 1 ---
CÂU GỐC (ĐẦU VÀO):
sáng 12 10   người nhà sản phụ nguyễn thị thu hà   27 tuổi   ở tp huế   thừa thiên   huế   đã tụ tập trước khoa sản   bệnh viện trung ương huế   để yêu cầu làm rõ nguyên nhân cái chết bất thường của chị hà   theo anh nguyễn xuân phước   anh trai sản phụ     chị hà nhập viện ngày 5 10 trong tình trạng thai 9 tháng   huyết áp cao   có dấu hiệu thiếu máu   sau khi nhập viện   các bác sĩ bệnh viện trung ương huế chẩn đoán sản phụ nhiễm sán lá gan   được truyền máu và tiểu cầu   chị hà khoẻ lại   đến ...

BẢN TÓM TẮT CHUẨN:
sau ca mổ sinh   chị hà sức khoẻ bình thường nhưng sau đó 

## 6. Lưu mô hình xuống Ổ cứng (Save / Load)
Trích xuất toàn bộ Trọng số (Weights) và lưu thành file `.pth`. Lần mở máy sau chỉ cần `load_state_dict()` là dùng được ngay, không phải train lại.

In [8]:
# Lưu trọng số xuống ổ cứng
torch.save(encoder.state_dict(), 'encoder.pth')
torch.save(decoder.state_dict(), 'decoder.pth')
print('Đã lưu: encoder.pth và decoder.pth')

# ── Cách nạp lại lần sau ──────────────────────────────────────
# encoder_new = EncoderRNN(lang.n_words, hidden_size).to(device)
# encoder_new.load_state_dict(torch.load('encoder.pth'))
# encoder_new.eval()   # Tắt Dropout/BN – bật chế độ Inference
#
# decoder_new = AttnDecoderRNN(hidden_size, lang.n_words, max_length=MAX_LENGTH).to(device)
# decoder_new.load_state_dict(torch.load('decoder.pth'))
# decoder_new.eval()

Đã lưu: encoder.pth và decoder.pth


## 7. Đánh giá tự động bằng độ đo ROUGE (Quantitative Evaluation)
Sử dụng thư viện `rouge-score` để đo mức độ trùng lặp giữa văn bản do AI sinh ra (Prediction) và bản tóm tắt gốc (Reference).

In [11]:
%pip install -qq rouge-score

import pandas as pd
from rouge_score import rouge_scorer

print("Bắt đầu tính điểm ROUGE cho tập Test...")
try:
    df_res = pd.read_csv('test_predictions.csv')
    df_res = df_res.dropna()
    
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
    rouge1_f, rouge2_f, rougeL_f = [], [], []

    for idx, row in df_res.iterrows():
        ref = str(row['summary'])
        pred = str(row['prediction'])
        
        scores = scorer.score(ref, pred)
        rouge1_f.append(scores['rouge1'].fmeasure)
        rouge2_f.append(scores['rouge2'].fmeasure)
        rougeL_f.append(scores['rougeL'].fmeasure)

    print(f"\n[ KẾT QUẢ ROUGE TRUNG BÌNH TRÊN {len(df_res)} BÀI TEST ]")
    print(f"ROUGE-1 F1: {sum(rouge1_f)/len(rouge1_f):.4f} (Độ trùng lặp từng từ)")
    print(f"ROUGE-2 F1: {sum(rouge2_f)/len(rouge2_f):.4f} (Độ trùng lặp cụm 2 từ liền kề)")
    print(f"ROUGE-L F1: {sum(rougeL_f)/len(rougeL_f):.4f} (Độ trùng lặp chuỗi con dài nhất)")
    
    # So sánh thực tế
    print("\nNhận xét: Với mô hình LSTM đào tạo trên lượng dữ liệu ít, điểm ROUGE thường khá thấp (dưới 0.3).")
    print("Để đạt điểm ROUGE cao (>0.4), người ta thường chuyển sang dùng các mô hình Thế hệ 3, 4 (như PhoBERT, ViT5).")
    
except FileNotFoundError:
    print("Lỗi: Không tìm thấy file 'test_predictions.csv'. Bạn cần chạy ô kiểm thử ở trên trước!")

Note: you may need to restart the kernel to use updated packages.
Bắt đầu tính điểm ROUGE cho tập Test...

[ KẾT QUẢ ROUGE TRUNG BÌNH TRÊN 292 BÀI TEST ]
ROUGE-1 F1: 0.2090 (Độ trùng lặp từng từ)
ROUGE-2 F1: 0.0343 (Độ trùng lặp cụm 2 từ liền kề)
ROUGE-L F1: 0.1498 (Độ trùng lặp chuỗi con dài nhất)

Nhận xét: Với mô hình LSTM đào tạo trên lượng dữ liệu ít, điểm ROUGE thường khá thấp (dưới 0.3).
Để đạt điểm ROUGE cao (>0.4), người ta thường chuyển sang dùng các mô hình Thế hệ 3, 4 (như PhoBERT, ViT5).
